In [19]:
using Pkg
Pkg.activate("./insight_calc")
using D4M, DataFrames, DotEnv
include("insight_calc/src/automata.jl")
include("insight_calc/src/services.jl")
include("insight_calc/src/storage.jl")

  Activating project at `~/populi.Wk/InsightCircle/insight_calc`
[ Info: ENV_PATH==/home/gcr/populi.Wk/InsightCircle/.env


_rowsToAssocEAV (generic function with 1 method)

In [20]:
Pkg.status()

Project InsightCalc v0.1.0
Status `~/populi.Wk/InsightCircle/insight_calc/Project.toml`
  [67c07d97] Automa v1.1.0
  [ca196bdc] D4M v0.6.11 `../../D4M.jl`
  [a93c6f00] DataFrames v1.8.1
  [4dc1fcf4] DotEnv v1.0.0
  [55e21f81] GoogleCloud v0.11.0
  [cd3eb016] HTTP v1.11.0
  [0f8b85d8] JSON3 v1.14.3
⌃ [df9a0d86] Oxygen v1.7.5
  [856f2bd8] StructTypes v1.11.0
  [ade2ca70] Dates v1.11.0
  [56ddb016] Logging v1.11.0
  [44cfe95a] Pkg v1.11.0
  [8dfed614] Test v1.11.0
Info Packages marked with ⌃ have new versions available and may be upgradable.


In [21]:
session = getServer()
fieldnames(typeof(session))

(:project, :dataset, :session)

In [22]:
listTables()

[ Info: Session is unauthorized. Fetching new token...


1-element Vector{String}:
 "yt_metadata"

In [23]:
srv     = getServer()
jobId   = _submitJob(srv.session, srv.project, """
    SELECT id, COUNT(*) AS n
    FROM `creator-d4m-2026-1774038056.insight_metadata.yt_metadata`
    GROUP BY id
    HAVING COUNT(*) > 1
    ORDER BY n DESC
""")
url     = "$_BQ_BASE/projects/$(srv.project)/queries/$jobId?maxResults=100&location=us-central1"
payload = JSON3.read(HTTP.get(url, _authHeaders(srv.session)).body)

isempty(get(payload, :rows, [])) ? "No duplicates" : begin
    cols = [String(f.name) for f in payload.schema.fields]
    rows = [[isnothing(c.v) ? "" : String(c.v) for c in row.f] for row in payload.rows]
    DataFrame(permutedims(hcat(rows...)), cols)
end


"No duplicates"

In [24]:
srv     = getServer()
jobId   = _submitJob(srv.session, srv.project, """
    SELECT id, query_term
    FROM `creator-d4m-2026-1774038056.insight_metadata.yt_metadata`
""")
url     = "$_BQ_BASE/projects/$(srv.project)/queries/$jobId?maxResults=10000&location=us-central1"
payload = JSON3.read(HTTP.get(url, _authHeaders(srv.session)).body)


JSON3.Object{Vector{UInt8}, Vector{UInt64}} with 9 entries:
  :kind                => "bigquery#getQueryResultsResponse"
  :etag                => "HsR+j4L/ADk+lMfJumo8kg=="
  :schema              => {…
  :jobReference        => {…
  :totalRows           => "2339"
  :rows                => Object[{…
  :totalBytesProcessed => "0"
  :jobComplete         => true
  :cacheHit            => true

In [25]:
using DataFrames

aa = queryYtMetadata(":")

rows, cols, vals = find(aa)
lookup   = Dict(zip(zip(rows, cols), vals))
all_ids  = unique(rows)
all_cols = unique(cols)

DataFrame(
    "id" => all_ids,
    [c => [get(lookup, (id, c), "") for id in all_ids] for c in all_cols]...
)


Row,id,duration,query_term_1,query_term_2,query_term_3,query_term_4,subscribers,title,uploader,views
,String,String,String,String,String,String,String,String,String,String
1,-134qFqjMpo,592,natural language processing,,,,,Natural Language Processing in TAMIL (NLP என்றால் என்ன?),Shriram Vasudevan,53502
2,-33k3Pa-xig,157,cloud computing,,,,,Cloud Computing and Distributed Systems Week 1 || NPTEL ANSWERS 2026 #nptel #nptel2026 #myswayam,MY SWAYAM,1520
3,-33oXx0TwHI,7105,natural language processing,,,,,Natural Language Processing (NLP) Full Course | Learn NLP in Deep Learning Tutorial,WsCube Tech,157241
4,-4E2-0sxVUM,670,computer vision,,,,,Computer Vision: Crash Course Computer Science #35,CrashCourse,474392
5,-4ZZMAJQnaQ,517,data science,,,,,Future of Data Science in India | Data Science Career India | Intellipaat,Intellipaat,2397
6,-7kv6jf0isQ,3700,reinforcement learning,,,,,Stanford CS224R Deep Reinforcement Learning | Spring 2025 | Lecture 6: Q-Learning,Stanford Online,4839
7,-8O32k26RWA,41596,cloud computing,,,,,Cloud Computing Full course | Cloud Computing Tutorial for Beginners in 2022 | Great Learning,Great Learning,144679
8,-9bo8HlSxwQ,2869,deep learning,,,,,CS50x 2026 - Artificial Intelligence,CS50,194486
9,-Bfc55rmFfM,1045,cloud computing,,,,,How to Become a Cloud Engineer in 2025 (Step-by-Step Roadmap),NextWork,59208


In [28]:
using DataFrames

aa1 = queryYtMetadata("-9bo8HlSxwQ..-IvNzmrcyUM, :")
aa2 = val2col(aa1,"|")
rows, cols, vals = find(aa2)
lookup   = Dict(zip(zip(rows, cols), vals))
all_ids  = unique(rows)
all_cols = unique(cols)

DataFrame(
    "id" => all_ids,
    [c => [get(lookup, (id, c), "") for id in all_ids] for c in all_cols]...
)


Row,id,duration|1045,duration|1417,duration|2210,duration|2869,duration|37437,query_term_1|cloud computing,query_term_1|data science,query_term_1|deep learning,query_term_1|kubernetes tutorial,query_term_1|machine learning tutorial,title|CS50x 2026 - Artificial Intelligence,title|Data Science Full Course - Learn Data Science in 10 Hours | Data Science For Beginners | Edureka,title|How to Become a Cloud Engineer in 2025 (Step-by-Step Roadmap),title|Simple Machine Learning Code Tutorial for Beginners with Sklearn Scikit-Learn,title|Storage and Backup in Kubernetes! // Longhorn Tutorial,uploader|CS50,uploader|Christian Lempa,uploader|NextWork,uploader|Python Simplified,uploader|edureka!,views|194486,views|45201,views|4589577,views|49152,views|59208
,String,Any,Any,Any,Any,Any,Any,Any,Any,Any,Any,Any,Any,Any,Any,Any,Any,Any,Any,Any,Any,Any,Any,Any,Any,Any
1,-Bfc55rmFfM,1,,,,,1,,,,,,,1,,,,,1,,,,,,,1
2,-IvNzmrcyUM,,1,,,,,,,,1,,,,1,,,,,1,,,,,1,
3,-ImtLXcEna8,,,1,,,,,,1,,,,,,1,,1,,,,,1,,,
4,-9bo8HlSxwQ,,,,1,,,,1,,,1,,,,,1,,,,,1,,,,
5,-ETQ97mXXF0,,,,,1,,1,,,,,1,,,,,,,,1,,,1,,


In [ ]:

rows, cols, vals = find(aa1')
lookup   = Dict(zip(zip(rows, cols), vals))
all_ids  = unique(rows)
all_cols = unique(cols)

DataFrame(
    "id" => all_ids,
    [c => [get(lookup, (id, c), "") for id in all_ids] for c in all_cols]...
)


In [ ]:
ee = aa1' * aa1

rows, cols, vals = find(ee)
lookup   = Dict(zip(zip(rows, cols), vals))
all_ids  = unique(rows)
all_cols = unique(cols)

DataFrame(
    "id" => all_ids,
    [c => [get(lookup, (id, c), "") for id in all_ids] for c in all_cols]...
)

In [ ]:
using DataFrames

aa = queryYtMetadata(":")


